# Train and evaluate DQN on Breakout with Google Colab CPU

This notebook runs the repository's existing DQN implementation on CPU. It does not claim a full reproduction of the paper: the default experiment uses a smaller replay buffer and 250,000 decisions. Training may take several hours. Colab can disconnect; periodic model checkpoints are written to Drive, but these are inference checkpoints and do not resume optimizer/replay state.

In Colab, select **Runtime → Change runtime type → CPU**. Run cells from top to bottom. The Atari ROM distributed with ALE is subject to its own license; use it only where permitted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

PROJECT_DIR = Path('/content/DQN')
if not (PROJECT_DIR / 'pyproject.toml').exists():
    !git clone https://github.com/bhagathchan1/DQN.git {PROJECT_DIR}
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
%pip install -e '.[env]'
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert not torch.cuda.is_available(), 'Switch Colab to a CPU runtime for this notebook.'

In [ ]:
# Check Gymnasium + ALE + Breakout before starting a long run.
%run scripts/environment_smoke_test.py

In [ ]:
# Keep durable files on Drive. Update RUN_ID to create a separate experiment.
from pathlib import Path
import shutil, yaml

DRIVE_DIR = Path('/content/drive/MyDrive/DQN')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for name in ('checkpoints', 'results'):
    local = PROJECT_DIR / name
    durable = DRIVE_DIR / name
    durable.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        shutil.move(str(local), str(durable / 'initial_repository_artifacts'))
    local.symlink_to(durable, target_is_directory=True)

RUN_ID = 'breakout_colab_cpu_seed7'
config = yaml.safe_load((PROJECT_DIR / 'configs/default.yaml').read_text())
config['run']['id'] = RUN_ID
config['training']['device'] = 'cpu'
# Adjust this before training if your Colab session has a shorter time budget.
config['training']['total_environment_steps'] = 250_000
config['training']['checkpoint_frequency'] = 25_000
config_path = PROJECT_DIR / 'configs/colab_cpu.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('Config:', config_path)
print('Checkpoints and metrics will persist in:', DRIVE_DIR)

In [ ]:
# Train. Every 25,000 decisions saves model weights to Google Drive.
!dqn-train --config configs/colab_cpu.yaml

In [ ]:
# Evaluate the final checkpoint and save the episode-level results on Drive.
checkpoint = DRIVE_DIR / 'checkpoints' / f'{RUN_ID}_final.pt'
evaluation = DRIVE_DIR / 'results/metrics' / f'{RUN_ID}_evaluation.json'
!dqn-evaluate --config configs/colab_cpu.yaml --checkpoint {checkpoint} --output {evaluation}
print('Evaluation:', evaluation)

## Inspect the run

Training episode rewards are in `MyDrive/DQN/results/metrics/`. The final weights are in `MyDrive/DQN/checkpoints/`. To compare against the paper, see `docs/upstream_comparison.md`; this CPU notebook reproduces the project's current compact experiment rather than the paper's 10-million-decision run. The GIFs and figures under `assets/upstream_dqn/` are original upstream reference assets, not outputs from this training run.